# 02 - Preprocessing
### Phishing URL Detector — PhiUSIIL Dataset

Goal: prepare a clean, model-ready feature matrix from the EDA output — drop identifier/raw-text columns, split into train/test sets, and scale numeric features where needed.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

df = pd.read_csv("../data/processed/eda_cleaned.csv")
print("Shape:", df.shape)
df.head()

Shape: (235370, 56)


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


## 1. Drop Identifier / Raw-Text Columns

These columns were flagged in the EDA notebook as unsuitable for direct model input:
- `FILENAME` — random file identifier, no signal
- `URL`, `Domain` — raw text (lexical signal is already captured in engineered numeric columns)
- `TLD` — raw text (numeric proxies `TLDLength`, `TLDLegitimateProb` already exist)
- `Title` — raw scraped text, high cardinality, not usable as-is

In [2]:
cols_to_drop = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df_model = df.drop(columns=cols_to_drop)
print("Columns dropped:", cols_to_drop)
print("Remaining shape:", df_model.shape)
df_model.head()

Columns dropped: ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
Remaining shape: (235370, 51)


,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,31,24,0,100.0,1.000000,0.522907,0.061933,3,1,0,...,0,0,1,34,20,28,119,0,124,1
1,23,16,0,100.0,0.666667,0.032650,0.050207,2,1,0,...,0,0,1,50,9,8,39,0,217,1
2,29,22,0,100.0,0.866667,0.028555,0.064129,2,2,0,...,0,0,1,10,2,7,42,2,5,1
3,26,19,0,100.0,1.000000,0.522907,0.057606,3,1,0,...,1,1,1,3,27,15,22,1,31,1
4,33,26,0,100.0,1.000000,0.079963,0.059441,3,1,0,...,1,0,1,244,15,34,72,1,85,1


## 2. Sanity Checks Before Splitting

In [3]:
# Confirm no missing values and all columns are numeric
print("Missing values:", df_model.isnull().sum().sum())
print("Non-numeric columns:", df_model.select_dtypes(exclude=[np.number]).columns.tolist())
print("Final feature count (excluding label):", df_model.shape[1] - 1)

Missing values: 0
Non-numeric columns: []
Final feature count (excluding label): 50


## 3. Train / Test Split

Stratified split on `label` to preserve the class ratio (~57% legitimate / 43% phishing) in both sets.

In [4]:
X = df_model.drop(columns=['label'])
y = df_model['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train label distribution:\n", y_train.value_counts(normalize=True))
print("Test label distribution:\n", y_test.value_counts(normalize=True))

Train shape: (188296, 50) Test shape: (47074, 50)
Train label distribution:
 label
1    0.572928
0    0.427072
Name: proportion, dtype: float64
Test label distribution:
 label
1    0.572928
0    0.427072
Name: proportion, dtype: float64


## 4. Feature Scaling

Tree-based models (Random Forest, XGBoost) don't strictly need scaling, but Logistic Regression does.
We fit the scaler on the training set only, then apply it to both sets, to avoid data leakage.

In [5]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaling complete. Example means (train, should be ~0):")
X_train_scaled.mean().head()

Scaling complete. Example means (train, should be ~0):


URLLength               7.939531e-17
DomainLength           -1.159232e-16
IsDomainIP             -4.830133e-18
URLSimilarityIndex      6.550868e-17
CharContinuationRate   -3.109398e-17
dtype: float64

## 5. Save Processed Data + Scaler

- Unscaled train/test sets are used for tree-based models (Random Forest, XGBoost)
- Scaled train/test sets are used for Logistic Regression
- The fitted scaler is saved so the same transformation can be applied later in the live `/scan` endpoint

In [6]:
out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

X_train.to_csv(f"{out_dir}/X_train.csv", index=False)
X_test.to_csv(f"{out_dir}/X_test.csv", index=False)
y_train.to_csv(f"{out_dir}/y_train.csv", index=False)
y_test.to_csv(f"{out_dir}/y_test.csv", index=False)

X_train_scaled.to_csv(f"{out_dir}/X_train_scaled.csv", index=False)
X_test_scaled.to_csv(f"{out_dir}/X_test_scaled.csv", index=False)

os.makedirs("../../backend/app/ml/saved_models", exist_ok=True)
joblib.dump(scaler, "../../backend/app/ml/saved_models/scaler.pkl")
joblib.dump(list(X_train.columns), "../../backend/app/ml/saved_models/feature_columns.pkl")

print("Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("Saved: X_train_scaled.csv, X_test_scaled.csv")
print("Saved: scaler.pkl, feature_columns.pkl -> backend/app/ml/saved_models/")

Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv
Saved: X_train_scaled.csv, X_test_scaled.csv
Saved: scaler.pkl, feature_columns.pkl -> backend/app/ml/saved_models/


## Summary

- Final feature count, train/test sizes, and class balance are confirmed above.
- Both unscaled and scaled versions are saved so each model type gets the input it needs.
- Next notebook (`03_baseline_models.ipynb`) will train Logistic Regression, Random Forest, and XGBoost on this data and compare results.